In [ ]:
!pip -q install gradio gTTS faster-whisper scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 63.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 1.16.1 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [ ]:
import os
import re
import random
import tempfile
from collections import Counter

import gradio as gr
from gtts import gTTS
from faster_whisper import WhisperModel

from sklearn.feature_extraction.text import TfidfVectorizer

import torch

print("✅ All libraries imported successfully!")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

✅ All libraries imported successfully!
PyTorch version: 2.11.0+cpu
GPU available: False


In [ ]:
# Automatically use GPU if available
if torch.cuda.is_available():
    device = "cuda"
    compute_type = "float16"
else:
    device = "cpu"
    compute_type = "int8"

print("Loading Whisper model...")
print("Device:", device)

whisper_model = WhisperModel(
    "base",
    device=device,
    compute_type=compute_type
)

print("✅ Speech recognition model loaded successfully!")

Loading Whisper model...
Device: cpu


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Speech recognition model loaded successfully!


In [ ]:
def speech_to_text(audio_file, language):
    if audio_file is None:
        return "⚠️ Please record or upload an audio file."

    try:
        if language == "English":
            selected_language = "en"
        elif language == "Tamil":
            selected_language = "ta"
        else:
            selected_language = None

        segments, info = whisper_model.transcribe(
            audio_file,
            language=selected_language,
            beam_size=5,
            vad_filter=True
        )

        text = " ".join(
            segment.text.strip()
            for segment in segments
        ).strip()

        if not text:
            return "⚠️ No speech detected. Please try again."

        return text

    except Exception as e:
        return f"❌ Speech recognition error: {str(e)}"


print("✅ Speech-to-text function ready!")

✅ Speech-to-text function ready!


In [ ]:
def text_to_speech(text, language):
    if not text or not text.strip():
        return None

    try:
        lang_code = "ta" if language == "Tamil" else "en"

        output_file = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".mp3"
        ).name

        speech = gTTS(
            text=text,
            lang=lang_code,
            slow=False
        )

        speech.save(output_file)

        return output_file

    except Exception as e:
        print("TTS Error:", e)
        return None


print("✅ Text-to-speech function ready!")

✅ Text-to-speech function ready!


In [ ]:
STOPWORDS = {
    "the", "is", "are", "was", "were", "and", "or", "of",
    "to", "in", "on", "for", "with", "a", "an", "as",
    "by", "from", "that", "this", "these", "those",
    "it", "its", "be", "been", "has", "have", "had",
    "will", "can", "may", "into", "than", "also",
    "about", "which", "their", "there", "they", "them",
    "you", "your", "we", "our", "he", "she", "his", "her",
    "இது", "அது", "மற்றும்", "ஒரு", "இந்த", "அந்த",
    "என்று", "உள்ளது", "ஆகும்"
}


def split_sentences(text):
    text = re.sub(r"\s+", " ", text.strip())

    sentences = re.split(
        r'(?<=[.!?।])\s+|[\n]+',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def extract_keywords(text, top_n=10):

    sentences = split_sentences(text)

    if not sentences:
        return []

    try:
        vectorizer = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b[\w\u0B80-\u0BFF]{3,}\b"
        )

        matrix = vectorizer.fit_transform(sentences)

        scores = matrix.sum(axis=0).A1
        terms = vectorizer.get_feature_names_out()

        ranked_indices = scores.argsort()[::-1]

        keywords = []

        for index in ranked_indices:

            word = terms[index]

            if word.lower() not in STOPWORDS:
                if word not in keywords:
                    keywords.append(word)

            if len(keywords) >= top_n:
                break

        return keywords

    except Exception:
        words = re.findall(
            r"[\w\u0B80-\u0BFF]{3,}",
            text.lower()
        )

        frequency = Counter(
            word for word in words
            if word not in STOPWORDS
        )

        return [
            word for word, count
            in frequency.most_common(top_n)
        ]


print("✅ Keyword extraction ready!")

✅ Keyword extraction ready!


In [ ]:
def generate_summary(text, sentence_count=3):

    if not text or not text.strip():
        return "⚠️ Please enter some study notes."

    sentences = split_sentences(text)

    if len(sentences) <= sentence_count:
        return " ".join(sentences)

    keywords = extract_keywords(
        text,
        top_n=12
    )

    scored_sentences = []

    for index, sentence in enumerate(sentences):

        sentence_lower = sentence.lower()

        score = 0

        for keyword in keywords:
            if keyword.lower() in sentence_lower:
                score += 1

        # Slight preference for meaningful sentence length
        if 40 <= len(sentence) <= 250:
            score += 0.5

        scored_sentences.append(
            (score, index, sentence)
        )

    best_sentences = sorted(
        scored_sentences,
        key=lambda x: x[0],
        reverse=True
    )[:sentence_count]

    # Restore original order
    best_sentences = sorted(
        best_sentences,
        key=lambda x: x[1]
    )

    summary = " ".join(
        sentence
        for score, index, sentence
        in best_sentences
    )

    return summary


print("✅ Summary generator ready!")

✅ Summary generator ready!


In [ ]:
def generate_quiz(text, number_of_questions=5):

    if not text or not text.strip():
        return "⚠️ Please enter study notes first.", ""

    sentences = split_sentences(text)
    keywords = extract_keywords(
        text,
        top_n=20
    )

    if len(keywords) < 4:
        return (
            "⚠️ Please provide more detailed notes "
            "to generate a quiz.",
            ""
        )

    questions = []
    used_keywords = set()

    for keyword in keywords:

        if len(questions) >= number_of_questions:
            break

        if keyword.lower() in used_keywords:
            continue

        matching_sentence = None

        for sentence in sentences:
            if keyword.lower() in sentence.lower():
                matching_sentence = sentence
                break

        if not matching_sentence:
            continue

        question_text = re.sub(
            re.escape(keyword),
            "__________",
            matching_sentence,
            count=1,
            flags=re.IGNORECASE
        )

        if question_text == matching_sentence:
            continue

        other_keywords = [
            k for k in keywords
            if k.lower() != keyword.lower()
            and k.lower() not in used_keywords
        ]

        if len(other_keywords) < 3:
            continue

        options = [
            keyword,
            other_keywords[0],
            other_keywords[1],
            other_keywords[2]
        ]

        random.shuffle(options)

        answer_letter = chr(
            65 + options.index(keyword)
        )

        question_number = len(questions) + 1

        question = (
            f"### Question {question_number}\n"
            f"**{question_text}**\n\n"
            f"A) {options[0]}\n\n"
            f"B) {options[1]}\n\n"
            f"C) {options[2]}\n\n"
            f"D) {options[3]}\n"
        )

        answer = (
            f"**{question_number}. "
            f"{answer_letter}) {keyword}**"
        )

        questions.append(
            (question, answer)
        )

        used_keywords.add(keyword.lower())

    if not questions:
        return (
            "⚠️ Could not generate a quiz from these notes.",
            ""
        )

    quiz_output = "\n\n---\n\n".join(
        q[0] for q in questions
    )

    answer_output = "\n".join(
        q[1] for q in questions
    )

    return quiz_output, answer_output


print("✅ Quiz generator ready!")

✅ Quiz generator ready!


In [ ]:
def analyze_notes(text):

    if not text or not text.strip():
        return (
            "⚠️ Please enter your study notes.",
            "⚠️ No keywords available."
        )

    summary = generate_summary(
        text,
        sentence_count=3
    )

    keywords = extract_keywords(
        text,
        top_n=10
    )

    keyword_display = " • ".join(
        f"**{word}**"
        for word in keywords
    )

    return summary, keyword_display


print("✅ Study analysis ready!")

✅ Study analysis ready!


In [ ]:
css = """
/* ---------- GLOBAL ---------- */

body {
    background:
        radial-gradient(circle at top left, #312e81 0%, transparent 35%),
        radial-gradient(circle at bottom right, #7c3aed 0%, transparent 35%),
        #0f172a !important;
}

.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    font-family: Inter, Arial, sans-serif !important;
}

/* ---------- HEADER ---------- */

.hero {
    padding: 42px 30px;
    margin-bottom: 25px;
    border-radius: 25px;
    text-align: center;
    background:
        linear-gradient(
            135deg,
            rgba(124,58,237,.95),
            rgba(30,64,175,.95)
        );
    box-shadow: 0 20px 50px rgba(0,0,0,.35);
    color: white;
}

.hero h1 {
    font-size: 42px;
    margin: 0;
    font-weight: 800;
}

.hero p {
    font-size: 17px;
    margin-top: 12px;
    opacity: .9;
}

/* ---------- CARDS ---------- */

.card {
    background: rgba(255,255,255,.08);
    backdrop-filter: blur(18px);
    border: 1px solid rgba(255,255,255,.14);
    border-radius: 20px;
    padding: 22px;
    margin-bottom: 18px;
}

/* ---------- BUTTONS ---------- */

button {
    border-radius: 12px !important;
    font-weight: 700 !important;
}

.primary-btn {
    min-height: 48px !important;
}

/* ---------- TEXTBOX ---------- */

textarea,
input {
    border-radius: 12px !important;
}

/* ---------- TABS ---------- */

.tab-nav button {
    font-weight: 700 !important;
}

/* ---------- FOOTER ---------- */

.footer {
    text-align: center;
    padding: 25px;
    color: #cbd5e1;
    font-size: 13px;
}

/* ---------- MOBILE ---------- */

@media (max-width: 700px) {

    .hero h1 {
        font-size: 30px;
    }

    .hero p {
        font-size: 14px;
    }
}
"""


with gr.Blocks(
    title="AI Study Buddy - Listen & Learn",
    css=css,
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="indigo"
    )
) as demo:

    # HEADER
    gr.HTML("""
    <div class="hero">
        <h1>🧠 AI Study Buddy</h1>
        <p>Listen & Learn — Your intelligent study companion</p>
        <p>
            🎤 Speech to Text &nbsp; • &nbsp;
            🔊 Text to Speech &nbsp; • &nbsp;
            📝 Summary &nbsp; • &nbsp;
            🔑 Keywords &nbsp; • &nbsp;
            ❓ Quiz
        </p>
    </div>
    """)

    # LANGUAGE
    with gr.Row():

        language = gr.Dropdown(
            choices=["English", "Tamil"],
            value="English",
            label="🌐 Study Language"
        )

    # TABS
    with gr.Tabs():

        # --------------------------------
        # TAB 1 : STUDY NOTES
        # --------------------------------

        with gr.Tab("📚 Study Notes"):

            gr.Markdown(
                "## 📝 Enter Your Study Material"
            )

            notes_input = gr.Textbox(
                label="Study Notes",
                placeholder=(
                    "Paste your study notes here...\n\n"
                    "Example: Cloud computing is the delivery "
                    "of computing services over the internet."
                ),
                lines=12
            )

            with gr.Row():

                analyze_button = gr.Button(
                    "🧠 Analyze Notes",
                    variant="primary",
                    elem_classes="primary-btn"
                )

                clear_button = gr.Button(
                    "🗑️ Clear"
                )

            gr.Markdown("## 📌 Short Summary")

            summary_output = gr.Textbox(
                label="AI Generated Summary",
                lines=6,
                interactive=False
            )

            gr.Markdown("## 🔑 Important Keywords")

            keywords_output = gr.Markdown(
                "Your important keywords will appear here."
            )


            analyze_button.click(
                fn=analyze_notes,
                inputs=notes_input,
                outputs=[
                    summary_output,
                    keywords_output
                ]
            )

            clear_button.click(
                fn=lambda: ("", "", ""),
                inputs=[],
                outputs=[
                    notes_input,
                    summary_output,
                    keywords_output
                ]
            )


        # --------------------------------
        # TAB 2 : SPEECH TO TEXT
        # --------------------------------

        with gr.Tab("🎤 Speech → Text"):

            gr.Markdown(
                "## 🎤 Speak Your Notes"
            )

            gr.Markdown(
                "Record your voice or upload an audio file. "
                "The AI will convert your speech into text."
            )

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎙️ Record / Upload Audio"
            )

            speech_button = gr.Button(
                "🎤 Convert Speech to Text",
                variant="primary"
            )

            speech_output = gr.Textbox(
                label="📝 Transcribed Text",
                lines=10,
                interactive=False
            )

            speech_button.click(
                fn=speech_to_text,
                inputs=[
                    audio_input,
                    language
                ],
                outputs=speech_output
            )


        # --------------------------------
        # TAB 3 : TEXT TO SPEECH
        # --------------------------------

        with gr.Tab("🔊 Listen & Learn"):

            gr.Markdown(
                "## 🔊 Listen to Your Study Material"
            )

            listen_text = gr.Textbox(
                label="Text to Read Aloud",
                placeholder="Enter your notes here...",
                lines=10
            )

            listen_button = gr.Button(
                "🔊 Generate Audio",
                variant="primary"
            )

            audio_output = gr.Audio(
                label="🎧 Your Study Audio",
                type="filepath"
            )

            listen_button.click(
                fn=text_to_speech,
                inputs=[
                    listen_text,
                    language
                ],
                outputs=audio_output
            )


        # --------------------------------
        # TAB 4 : QUIZ
        # --------------------------------

        with gr.Tab("❓ Smart Quiz"):

            gr.Markdown(
                "## 🧠 Generate a Quiz From Your Notes"
            )

            quiz_notes = gr.Textbox(
                label="Study Material",
                placeholder="Paste your notes here...",
                lines=10
            )

            question_count = gr.Slider(
                minimum=3,
                maximum=10,
                value=5,
                step=1,
                label="Number of Questions"
            )

            quiz_button = gr.Button(
                "✨ Generate Quiz",
                variant="primary"
            )

            quiz_output = gr.Markdown(
                "Your generated quiz will appear here."
            )

            gr.Markdown("## 🔐 Answer Key")

            answer_output = gr.Markdown(
                "Answer key will appear here."
            )

            quiz_button.click(
                fn=generate_quiz,
                inputs=[
                    quiz_notes,
                    question_count
                ],
                outputs=[
                    quiz_output,
                    answer_output
                ]
            )


    # FOOTER
    gr.HTML("""
    <div class="footer">
        🧠 AI Study Buddy — Listen & Learn<br>
        Built with Python, Whisper, NLP, gTTS & Gradio
    </div>
    """)

print("✅ Professional UI created successfully!")

/tmp/ipykernel_2364/2966703613.py:105: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Professional UI created successfully!


In [ ]:
print("🚀 Starting AI Study Buddy...")
print("⏳ Please wait for the public link...")

demo.launch(
    share=True,
    debug=True
)

🚀 Starting AI Study Buddy...
⏳ Please wait for the public link...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e7581f6251193c764b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
